# Create Chatbot with Message History Using Langchain

## Introduction

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model='Gemma2-9b-It',groq_api_key=groq_api_key)

In [3]:
# How to chat with LLM
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi My name is Hanif, I'am Data Engineer")])

AIMessage(content="Hi Hanif, it's nice to meet you!\n\nIt's great that you're a Data Engineer. That's a really important and exciting field right now. \n\nWhat kind of work do you do as a Data Engineer? Are you working on any interesting projects at the moment?  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 21, 'total_tokens': 88, 'completion_time': 0.121818182, 'prompt_time': 0.002099519, 'queue_time': 0.241797392, 'total_time': 0.123917701}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--92307c36-ad9e-411d-a626-391627902002-0', usage_metadata={'input_tokens': 21, 'output_tokens': 67, 'total_tokens': 88})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi My name is Hanif, I'am Data Engineer"),
        AIMessage("Hello Hanif, it's nice to meet you!\n\nA data engineer, eh? That's a fascinating field. What kind of projects are you working on these days?\n\nI'm always interested in learning more about how people use data to solve problems and make the world a better place.\n"),
        HumanMessage("Do you know my name? what my job?")
    ]
    )

AIMessage(content="Yes!  Even though I don't have memory of past conversations, you introduced yourself as Hanif, a Data Engineer, at the beginning of this chat. \n\nIs there anything else I can help you with today, Hanif? 😊  Perhaps you'd like to discuss some data engineering challenges you're facing, or maybe you're curious about the latest trends in the field?  I'm here to listen and chat.  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 104, 'total_tokens': 200, 'completion_time': 0.174545455, 'prompt_time': 0.005091629, 'queue_time': 0.24550220900000003, 'total_time': 0.179637084}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--493753bb-70fe-4302-88e9-d5a997dfd7fc-0', usage_metadata={'input_tokens': 104, 'output_tokens': 96, 'total_tokens': 200})

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

message_history = RunnableWithMessageHistory(model,get_session_history)

In [6]:
config = {'configurable':{'session_id':'chat_1'}}

In [7]:
message_history.invoke(
    [HumanMessage(content="Hi My name is Hanif, I'am Data Engineer")],
    config=config
)

AIMessage(content="Hi Hanif, it's nice to meet you! \n\nIt's great that you're a Data Engineer. That's a really important and interesting field. \n\nWhat kind of projects are you working on these days?\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 21, 'total_tokens': 73, 'completion_time': 0.094545455, 'prompt_time': 0.002169529, 'queue_time': 0.24456307, 'total_time': 0.096714984}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--166c06c8-731b-4a7b-9fc8-d6ee6bb4341f-0', usage_metadata={'input_tokens': 21, 'output_tokens': 52, 'total_tokens': 73})

In [8]:
message_history.invoke(
    [HumanMessage(content="Do you know my name? what my job?")],
    config=config
)

AIMessage(content='You\'re right! As a large language model, I don\'t have memory of past conversations.  \n\nSo, even though you told me your name is Hanif and that you\'re a Data Engineer, I don\'t actually "know" that information.  Every time we interact, it\'s like a fresh start.  \n\nWould you like to tell me more about your work as a Data Engineer?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 91, 'total_tokens': 180, 'completion_time': 0.161818182, 'prompt_time': 0.00437932, 'queue_time': 0.245296639, 'total_time': 0.166197502}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--b6fb944f-edf7-422e-8c6a-1135c71eac33-0', usage_metadata={'input_tokens': 91, 'output_tokens': 89, 'total_tokens': 180})

## Chat Prompt Template

In [9]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages(
    [
        ('system','you are research and innovation Skincare assistant for {skin_care_division}, you will give me a suggestion of my product based on your knowledge'),
        MessagesPlaceholder(variable_name='message')
    ]
)

chain = prompt | model

In [10]:
response = chain.invoke(
    {'message':[HumanMessage(content = 'Hi my name is Hanif, I am Research and Innovation for Cosmax Indonesia')],'skin_care_division':'sun and emmulsion'}
    )

In [11]:
print(response)

content="Hi Hanif, it's great to meet you! \n\nI can definitely help you brainstorm some skincare product ideas for sunscreens and emulsions. To give you the best suggestions, tell me more about what you're looking for. \n\nConsider these factors:\n\n* **Target audience:**  Who are you developing these products for? (e.g., young adults, mature skin, men, specific skin concerns like acne or hyperpigmentation)\n* **Unique selling points:** What makes your products stand out from the competition? (e.g., natural ingredients, innovative technology, specific benefits like anti-aging or brightening)\n* **Formulations:**  What type of textures and finishes are you interested in? (e.g., lightweight lotion, rich cream, gel, spray, matte finish, dewy finish)\n* **Sustainability:** Are there any eco-friendly or ethical considerations you want to incorporate? (e.g., recyclable packaging, cruelty-free ingredients, locally sourced materials)\n\nThe more information you provide, the more tailored and 

In [12]:
message_history = RunnableWithMessageHistory(
    chain
    ,get_session_history
    ,input_messages_key = 'message'
    )

In [13]:
config = {'configurable':{'session_id':'chat_skincare'}}

In [14]:
response = message_history.invoke(
    {'message':HumanMessage(content = 'Hi my name is Wia, I am Research and Innovation for Cosmax Indonesia'),'skin_care_division':'sun and emmulsion'},
    config=config
)

In [15]:
response.content

"Hello Wia! It's great to connect. \n\nAs your Skincare Research and Innovation assistant, I can definitely help brainstorm some sun and emulsion product ideas for Cosmax Indonesia. To give you the best suggestions, I need a little more information about your vision. \n\nTell me about:\n\n* **Target Market:** Who are you developing these products for? (e.g., young adults, mature skin, specific concerns like acne or hyperpigmentation)\n* **Product Focus:** \n    * **Sunscreen:** What type of protection are you looking for? (e.g., broad spectrum, SPF 30, SPF 50+, mineral vs. chemical filters) \n    * **Emulsion:** What key benefits do you want to highlight? (e.g., hydration, anti-aging, brightening, oil control)\n* **Unique Selling Point (USP):** What will make your products stand out from the competition? (e.g., natural ingredients, innovative technology, specific skin concerns addressed)\n* **Price Point:**  Are you aiming for a premium, mid-range, or accessible price point?\n\nOnce I 

In [16]:
message_history.invoke(
    {'message':HumanMessage(content = 'What is my name and job?'),'skin_care_division':'sun and emmulsion'},
    config=config
)

AIMessage(content="You are Wia, the Research and Innovation for Cosmax Indonesia.  \n\nIt's nice to be able to put a name and face to the person I'm helping! 😄  \n\nWhat can I do to assist you today with your sun and emulsion product ideas?  \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 348, 'total_tokens': 408, 'completion_time': 0.109090909, 'prompt_time': 0.012820329, 'queue_time': 0.24439205, 'total_time': 0.121911238}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--42a122a6-ae31-4449-9e0b-0d5e0fc7b956-0', usage_metadata={'input_tokens': 348, 'output_tokens': 60, 'total_tokens': 408})

## Manage Conversation History

In [17]:
from langchain_core.messages import SystemMessage,trim_messages

trimmer = trim_messages(
    max_tokens = 70,
    strategy = 'last',
    token_counter = model,
    include_system = True,
    allow_partial = False,
    start_on = 'human'
)

In [18]:
messages = [
    SystemMessage(content='you are research and innovation Skincare assistant for sun and emmulsion, you will give me a suggestion of my product based on your knowledge'),
    HumanMessage(content='Hi my name is Wia, I am Research and Innovation for Cosmax Indonesia'),
    AIMessage(content='Hello Wia, it\'s nice to meet you!\n\nA research and innovation specialist in skincare, eh? That\'s a fascinating field. What kind of projects are you working on these days?\n\nI\'m always interested in learning more about how people use data to solve problems and make the world a better place.\n'),
    HumanMessage(content='Currently I am on business trip in Cosmax korea. Do you have suggestion what must I do?'),
    AIMessage(content='You can learn new something about skincare in Cosmax Korea')   
]

In [19]:
trimmer.invoke(messages)

C:\Users\Hanif\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessage(content='you are research and innovation Skincare assistant for sun and emmulsion, you will give me a suggestion of my product based on your knowledge', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Currently I am on business trip in Cosmax korea. Do you have suggestion what must I do?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='You can learn new something about skincare in Cosmax Korea', additional_kwargs={}, response_metadata={})]

In [20]:
trimmer = trim_messages(
    max_tokens = 2000,
    strategy = 'last',
    token_counter = model,
    include_system = True,
    allow_partial = False,
    start_on = 'human'
)

In [21]:
trimmer.invoke(messages)

[SystemMessage(content='you are research and innovation Skincare assistant for sun and emmulsion, you will give me a suggestion of my product based on your knowledge', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi my name is Wia, I am Research and Innovation for Cosmax Indonesia', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello Wia, it's nice to meet you!\n\nA research and innovation specialist in skincare, eh? That's a fascinating field. What kind of projects are you working on these days?\n\nI'm always interested in learning more about how people use data to solve problems and make the world a better place.\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Currently I am on business trip in Cosmax korea. Do you have suggestion what must I do?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='You can learn new something about skincare in Cosmax Korea', additional_kwargs={}, response_metadata={})]

In [22]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter('message')|trimmer)
    | prompt | model

)

response = chain.invoke(
    {
        'message':messages + [HumanMessage(content='Who am I? what is my job and activity?')],
        'skin_care_division':'sun and emmulsion'
    } 
)

response.content

'You are Wia, a Research and Innovation specialist for Cosmax Indonesia.  \n\nYou are currently on a business trip to Cosmax Korea. \n\nYour job involves developing and improving skincare products, particularly focusing on sun care and emulsions.  \n\nYou are interested in learning new things about skincare while in Korea. \n\n\nLet me know if you want to brainstorm some ideas for your trip!\n'

In [23]:
message_history = RunnableWithMessageHistory(
    chain
    ,get_session_history
    ,input_messages_key = 'message'
    )

config = {'configurable':{'session_id':'chat_skincare'}}

In [24]:
response = chain.invoke(
    {
        'message':messages + [HumanMessage(content='Who am I? what is my job and activity?')],
        'skin_care_division':'sun and emmulsion'
    } ,
    config = config
)


In [25]:
response.content

"You are Wia, a Research and Innovation Specialist for Cosmax Indonesia.  \n\nYou're currently on a business trip in Cosmax Korea.  \n\nYour job involves developing and improving skincare products, particularly those related to sun protection and emulsions. \n\nThink about your goals for this trip! \n\nWhat are you hoping to learn or achieve while you're in Korea?  That will help me give you more specific suggestions.  \n\n\n\n"

## Working with VectorStore and Retriever

In [26]:
#PDF
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('data\MutasiBCA_010824-080824.pdf')
pdf_data = loader.load()
pdf_data

C:\Users\Hanif\AppData\Roaming\Python\Python311\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


[Document(metadata={'producer': 'Skia/PDF m82', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data\\MutasiBCA_010824-080824.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='MUTASI REKENING\nNO. REKENING\n: 575-0715-321\nNAMA\n: MUHAMMAD HANIF FAJ\nHALAMAN\n: 1/3\nJENIS TRANSAKSI\n: SEMUA\nPERIODE\n: 01/08/2024 - 08/08/2024\nMATA UANG\n: IDR\nCATATAN:\nApabila nasabah \x00dak melakukan sanggahan atas Laporan Mutasi Rekening ini sampai dengan akhir bulan\nberikutnya, nasabah dianggap telah menyetujui segala data yang tercantum pada Laporan Mutasi Rekening\nini.\nBCA berhak se\x00ap saat melakukan koreksi apabila ada kesalahan pada Laporan Mutasi Rekening\nTANGGAL\nKETERANGAN\nMUTASI\n08/08\nTGL: 0808         QR  014           00000.00GELAEL, MT \nTRANSAKSI DEBIT\n 64,624.00\nDB\n08/08\nTGL: 0808         QR  014           00000.00SHELL TEBE \nTRANSAKSI DEBIT\n 40,511.00\nDB\n07/08\n0708/FTFVA/WS9503139358/OVO         -                 -\n089611491724      \nTRSF E

In [29]:
#Vector Store
from langchain_chroma import Chroma

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")

In [31]:
llm = ChatGroq(groq_api_key=groq_api_key,model = "Llama3-8b-8192")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000011BF28EF9D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000011BF2D6BD90>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [32]:
llm.invoke('Who are you?')

AIMessage(content="I am LLaMA, an AI assistant developed by Meta AI that can understand and respond to human input in a conversational manner. I'm not a human, but a computer program designed to simulate conversation and answer questions to the best of my knowledge based on the data I've been trained on.\n\nMy training data includes a massive corpus of text from various sources, including books, articles, and online conversations. This allows me to understand and respond to a wide range of topics and questions.\n\nI'm designed to be helpful and informative, and I can assist with tasks such as:\n\n* Answering questions on various topics, including science, history, and culture.\n* Providing definitions and explanations for words and concepts.\n* Generating text on a given topic or subject.\n* Translation between languages.\n* And more!\n\nI'm constantly learning and improving, so please bear with me if I make any mistakes. I'm here to help and provide assistance to the best of my abilit

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

In [34]:
vector_store = Chroma.from_documents(pdf_data,embedding=embeddings)

In [36]:
vector_store.similarity_search('FamilyMart')

[Document(id='bd01e4b9-7472-421b-a5f8-9a3cc2162353', metadata={'creator': 'PyPDF', 'total_pages': 3, 'page_label': '2', 'page': 1, 'creationdate': '', 'source': 'data\\MutasiBCA_010824-080824.pdf', 'producer': 'Skia/PDF m82'}, page_content='MUTASI REKENING\nNO. REKENING\n: 575-0715-321\nNAMA\n: MUHAMMAD HANIF FAJ\nHALAMAN\n: 2/3\nJENIS TRANSAKSI\n: SEMUA\nPERIODE\n: 01/08/2024 - 08/08/2024\nMATA UANG\n: IDR\nCATATAN:\nApabila nasabah \x00dak melakukan sanggahan atas Laporan Mutasi Rekening ini sampai dengan akhir bulan\nberikutnya, nasabah dianggap telah menyetujui segala data yang tercantum pada Laporan Mutasi Rekening\nini.\nBCA berhak se\x00ap saat melakukan koreksi apabila ada kesalahan pada Laporan Mutasi Rekening\nTANGGAL\nKETERANGAN\nMUTASI\n04/08\n0308/FTFVA/WS9503170001/GOPAY TOPUP -                 -\n081384389400      \nTRSF E-BANKING DB\n 18,000.00\nDB\n03/08\nTGL: 0803         QR  915           00000.00FamilyMart \nTRANSAKSI DEBIT\n 3,800.00\nDB\n03/08\n0308/FTFVA/WS950317

In [37]:
#Async Query
await vector_store.asimilarity_search('FamilyMart')

[Document(id='bd01e4b9-7472-421b-a5f8-9a3cc2162353', metadata={'page': 1, 'source': 'data\\MutasiBCA_010824-080824.pdf', 'creator': 'PyPDF', 'producer': 'Skia/PDF m82', 'creationdate': '', 'page_label': '2', 'total_pages': 3}, page_content='MUTASI REKENING\nNO. REKENING\n: 575-0715-321\nNAMA\n: MUHAMMAD HANIF FAJ\nHALAMAN\n: 2/3\nJENIS TRANSAKSI\n: SEMUA\nPERIODE\n: 01/08/2024 - 08/08/2024\nMATA UANG\n: IDR\nCATATAN:\nApabila nasabah \x00dak melakukan sanggahan atas Laporan Mutasi Rekening ini sampai dengan akhir bulan\nberikutnya, nasabah dianggap telah menyetujui segala data yang tercantum pada Laporan Mutasi Rekening\nini.\nBCA berhak se\x00ap saat melakukan koreksi apabila ada kesalahan pada Laporan Mutasi Rekening\nTANGGAL\nKETERANGAN\nMUTASI\n04/08\n0308/FTFVA/WS9503170001/GOPAY TOPUP -                 -\n081384389400      \nTRSF E-BANKING DB\n 18,000.00\nDB\n03/08\nTGL: 0803         QR  915           00000.00FamilyMart \nTRANSAKSI DEBIT\n 3,800.00\nDB\n03/08\n0308/FTFVA/WS950317

In [38]:
vector_store.similarity_search_with_score('FamilyMart')

[(Document(id='bd01e4b9-7472-421b-a5f8-9a3cc2162353', metadata={'source': 'data\\MutasiBCA_010824-080824.pdf', 'creator': 'PyPDF', 'producer': 'Skia/PDF m82', 'total_pages': 3, 'creationdate': '', 'page': 1, 'page_label': '2'}, page_content='MUTASI REKENING\nNO. REKENING\n: 575-0715-321\nNAMA\n: MUHAMMAD HANIF FAJ\nHALAMAN\n: 2/3\nJENIS TRANSAKSI\n: SEMUA\nPERIODE\n: 01/08/2024 - 08/08/2024\nMATA UANG\n: IDR\nCATATAN:\nApabila nasabah \x00dak melakukan sanggahan atas Laporan Mutasi Rekening ini sampai dengan akhir bulan\nberikutnya, nasabah dianggap telah menyetujui segala data yang tercantum pada Laporan Mutasi Rekening\nini.\nBCA berhak se\x00ap saat melakukan koreksi apabila ada kesalahan pada Laporan Mutasi Rekening\nTANGGAL\nKETERANGAN\nMUTASI\n04/08\n0308/FTFVA/WS9503170001/GOPAY TOPUP -                 -\n081384389400      \nTRSF E-BANKING DB\n 18,000.00\nDB\n03/08\nTGL: 0803         QR  915           00000.00FamilyMart \nTRANSAKSI DEBIT\n 3,800.00\nDB\n03/08\n0308/FTFVA/WS95031

In [40]:
from typing import List
from langchain_core.runnables import RunnableLambda 
retriever  = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(['FamilyMart','Shell'])

[[Document(id='bd01e4b9-7472-421b-a5f8-9a3cc2162353', metadata={'source': 'data\\MutasiBCA_010824-080824.pdf', 'producer': 'Skia/PDF m82', 'creationdate': '', 'total_pages': 3, 'page_label': '2', 'creator': 'PyPDF', 'page': 1}, page_content='MUTASI REKENING\nNO. REKENING\n: 575-0715-321\nNAMA\n: MUHAMMAD HANIF FAJ\nHALAMAN\n: 2/3\nJENIS TRANSAKSI\n: SEMUA\nPERIODE\n: 01/08/2024 - 08/08/2024\nMATA UANG\n: IDR\nCATATAN:\nApabila nasabah \x00dak melakukan sanggahan atas Laporan Mutasi Rekening ini sampai dengan akhir bulan\nberikutnya, nasabah dianggap telah menyetujui segala data yang tercantum pada Laporan Mutasi Rekening\nini.\nBCA berhak se\x00ap saat melakukan koreksi apabila ada kesalahan pada Laporan Mutasi Rekening\nTANGGAL\nKETERANGAN\nMUTASI\n04/08\n0308/FTFVA/WS9503170001/GOPAY TOPUP -                 -\n081384389400      \nTRSF E-BANKING DB\n 18,000.00\nDB\n03/08\nTGL: 0803         QR  915           00000.00FamilyMart \nTRANSAKSI DEBIT\n 3,800.00\nDB\n03/08\n0308/FTFVA/WS95031

In [41]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs = {"k":1}
)
retriever.batch(['FamilyMart','Shell'])

[[Document(id='bd01e4b9-7472-421b-a5f8-9a3cc2162353', metadata={'total_pages': 3, 'page': 1, 'creator': 'PyPDF', 'creationdate': '', 'page_label': '2', 'producer': 'Skia/PDF m82', 'source': 'data\\MutasiBCA_010824-080824.pdf'}, page_content='MUTASI REKENING\nNO. REKENING\n: 575-0715-321\nNAMA\n: MUHAMMAD HANIF FAJ\nHALAMAN\n: 2/3\nJENIS TRANSAKSI\n: SEMUA\nPERIODE\n: 01/08/2024 - 08/08/2024\nMATA UANG\n: IDR\nCATATAN:\nApabila nasabah \x00dak melakukan sanggahan atas Laporan Mutasi Rekening ini sampai dengan akhir bulan\nberikutnya, nasabah dianggap telah menyetujui segala data yang tercantum pada Laporan Mutasi Rekening\nini.\nBCA berhak se\x00ap saat melakukan koreksi apabila ada kesalahan pada Laporan Mutasi Rekening\nTANGGAL\nKETERANGAN\nMUTASI\n04/08\n0308/FTFVA/WS9503170001/GOPAY TOPUP -                 -\n081384389400      \nTRSF E-BANKING DB\n 18,000.00\nDB\n03/08\nTGL: 0803         QR  915           00000.00FamilyMart \nTRANSAKSI DEBIT\n 3,800.00\nDB\n03/08\n0308/FTFVA/WS95031

In [42]:
message = """ 
Answer the question based on context only.

{question}

Context:

{context}


"""

prompt = ChatPromptTemplate.from_messages(["human",message])

rag_chain = {"context":retriever,"question":RunnablePassthrough()}|prompt| llm

response = rag_chain.invoke('how many I spend in FamilyMart? ')

In [44]:
response

AIMessage(content='Based on the context, it seems that the person spent 3,800.00 IDR at FamilyMart.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1047, 'total_tokens': 1070, 'completion_time': 0.038630257, 'prompt_time': 0.117771926, 'queue_time': 0.27200404199999995, 'total_time': 0.156402183}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_6d1c2b0784', 'finish_reason': 'stop', 'logprobs': None}, id='run--b0f2f167-4d66-4175-9459-6354b2dd17b0-0', usage_metadata={'input_tokens': 1047, 'output_tokens': 23, 'total_tokens': 1070})